# Heart Disease Prediction — Full Analysis

This notebook is the single entry point for the full project workflow. Run all cells **top-to-bottom** and everything will execute: data loading, EDA, model training, evaluation, and the Streamlit web app.

**Before opening this notebook**, make sure you have set up the virtual environment and launched Jupyter from inside it.

### First-time setup

```bash
# From the project root (ProjectEC3/)

# If you need to recreate the venv (e.g. wrong Python version):
rm -rf .venv

# Create venv with Python 3.11 (recommended — matches CI)
py -3.11 -m venv .venv

# Activate
# Windows:
.venv\Scripts\Activate.ps1
# macOS / Linux:
source .venv/bin/activate

# Install dependencies
pip install -r requirements.txt

# Launch Jupyter
jupyter notebook notebooks/analysis.ipynb
```

Once Jupyter opens, select the `.venv` kernel in the top right corner, then click **Run All**.

**Contents:**
1. Environment check
2. Package check & auto-install
3. Imports & path setup
4. Data loading and cleaning
5. Exploratory Data Analysis (EDA)
6. Model training
7. Evaluation & visualisations
8. Streamlit app — launch & inline results
9. Ethical reflection
10. Terminal app
11. CI/CD — GitHub Actions

---
## 1. Environment Check

This cell verifies that the notebook is running inside the project's `.venv` virtual environment and that the Python version is compatible. If the path shown does not contain `.venv`, stop here and relaunch Jupyter from the activated environment.

In [ ]:
import sys

print(f"Python executable : {sys.executable}")
print(f"Python version    : {sys.version}")

if ".venv" not in sys.executable and "venv" not in sys.executable:
    print("\n⚠️  WARNING: This kernel does not appear to be running inside a virtual environment.")
    print("   Activate .venv and relaunch Jupyter to ensure the correct packages are used.")
else:
    print("\n✅ Virtual environment detected — you're good to go.")
# Please run the foloowing in Git Bash if virtual env is not present:
# Close the notebook, then create the venv first:
# cd ~/git/ProjectEC4
# py -3.11 -m venv .venv
# source .venv/Scripts/activate
# pip install -r requirements.txt
# pip install ipykernel
# python -m ipykernel install --user --name=ProjectEC4 --display-name "Python (ProjectEC4)"

In [ ]:
# Verify all required packages are installed.
# If anything is missing, installs from requirements.txt automatically.
import importlib
import subprocess

required = ["numpy", "pandas", "sklearn", "matplotlib", "seaborn", "joblib", "streamlit"]
missing = [pkg for pkg in required if importlib.util.find_spec(pkg) is None]

if missing:
    print(f"⚠️  Missing packages: {missing}")
    print("Installing from requirements.txt...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r",
                           str(Path(PROJECT_ROOT) / "requirements.txt")])
    print("✅ Done — restart the kernel and run again.")
else:
    print("✅ All required packages found.")

---
## 2. Data Preparation

Before loading data we run `src/data_preparation.py` to prepare the Cleveland Heart Disease dataset.

This step:
- Loads `processed.cleveland.data` (303 rows, 14 columns)
- Converts target to binary (0 = no disease, 1+ = disease present)
- Imputes 6 missing values in `ca` and `thal` with column medians
- Saves to `data/heart_combined.csv`

The Cleveland dataset is used because it is the only UCI hospital dataset with reliable
measurements for all 13 features including `ca` and `thal` — the two most important
predictive features (~28% combined importance). See the Investigation Findings section
in README.md for the full analysis.

The original file is never modified. Run this cell once — subsequent runs are safe.

In [ ]:
# Run data preparation pipeline
# Loads processed.cleveland.data, imputes 6 missing values,
# converts target to binary and saves to heart_combined.csv
import subprocess
import sys
import os
from pathlib import Path

PROJECT_ROOT = Path(os.path.abspath('__file__')).parent.parent

result = subprocess.run(
    [sys.executable, str(PROJECT_ROOT / 'src' / 'data_preparation.py')],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('ERROR:', result.stderr)
else:
    print('✅ heart_combined.csv ready')


---
## 3. Imports & Path Setup

We resolve `PROJECT_ROOT` relative to this notebook's own location so imports work correctly regardless of where Jupyter was launched from. Then we import all libraries and configure paths used throughout the notebook.

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

# Resolve project root from this notebook's location (notebooks/ -> ..)
# Works correctly regardless of the directory Jupyter was launched from.
NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__"))
PROJECT_ROOT = os.path.abspath(os.path.join(NOTEBOOK_DIR, ".."))

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"Project root      : {PROJECT_ROOT}")
print(f"Notebook dir      : {NOTEBOOK_DIR}")

In [ ]:
import json
import socket
import subprocess
import time
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import HTML, IFrame, display
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
)

from src.data_processing import DataProcessor
from src.model_training import ModelTrainer

# Plotting defaults
plt.style.use("dark_background")
sns.set_theme(style="darkgrid", palette="bright")
plt.rcParams["figure.dpi"] = 110

# Central path constants — all file I/O uses these
DATA_PATH    = Path(PROJECT_ROOT) / "data" / "heart_combined.csv"
MODELS_DIR   = Path(PROJECT_ROOT) / "models"
MODEL_PATH   = MODELS_DIR / "random_forest.pkl"
RESULTS_PATH = MODELS_DIR / "training_results.json"

print("✅ All imports OK")
print(f"Dataset path : {DATA_PATH}  (exists: {DATA_PATH.exists()})")

---
## 4. Data Loading and Cleaning

We use `DataProcessor` from `src/data_processing.py` to load the UCI Heart Disease CSV and apply the project's standard cleaning steps: column name normalisation and missing value handling. After loading, we inspect the shape, the first few rows, and the class balance of the target variable.

In [ ]:
# Instantiate the DataProcessor and run load + clean
processor = DataProcessor(data_path=str(DATA_PATH))
processor.load_data()
processor.clean_data()

df = processor.df
print(f"Dataset shape  : {df.shape}  ({df.shape[0]} rows, {df.shape[1]} columns)")
print(f"Columns        : {df.columns.tolist()}")
df.head()

In [ ]:
# Built-in summary from DataProcessor: descriptive statistics for all columns
print("=== Dataset Summary ===")
processor.summary()

In [ ]:
# Data quality check: missing values and class balance
print("Missing values per column:")
print(df.isnull().sum())
print(f"\nTarget distribution (0 = no disease, 1 = disease):")
print(df["target"].value_counts())
print(f"\nClass balance: {df['target'].mean():.1%} positive (heart disease present)")

---
## 5. Exploratory Data Analysis (EDA)

Before training we explore the data visually to understand distributions, class separation, and relationships between features. This helps motivate feature selection and model choices.

We produce four sets of charts:
- **Target distribution** and **age histogram** split by target class
- **Numeric feature distributions** split by target
- **Categorical feature proportions** vs target
- **Correlation heatmap** across all features

In [ ]:
# --- Target distribution and age histogram side by side ---
# The left chart shows overall class balance (important: a heavily imbalanced
# dataset would require resampling). The right shows whether age alone
# separates the two classes.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

target_counts = df["target"].value_counts().sort_index()
axes[0].bar(["No Disease (0)", "Heart Disease (1)"], target_counts.values,
            color=["steelblue", "tomato"], edgecolor="white")
axes[0].set_title("Target Distribution")
axes[0].set_ylabel("Count")
for i, v in enumerate(target_counts.values):
    axes[0].text(i, v + 1, str(v), ha="center", fontweight="bold")

for label, color in [(0, "steelblue"), (1, "tomato")]:
    axes[1].hist(df[df["target"] == label]["age"], bins=20, alpha=0.6,
                 color=color, label=f"Target={label}")
axes[1].set_title("Age Distribution by Target")
axes[1].set_xlabel("Age")
axes[1].set_ylabel("Count")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# --- Numeric feature distributions split by target class ---
# Overlapping histograms reveal which continuous features discriminate well.
# For example, thalach (max heart rate) tends to be higher in disease-positive patients.
numeric_features = ["age", "trestbps", "chol", "thalach", "oldpeak"]
fig, axes = plt.subplots(1, len(numeric_features), figsize=(18, 4))

for ax, col in zip(axes, numeric_features):
    for label, color in [(0, "steelblue"), (1, "tomato")]:
        ax.hist(df[df["target"] == label][col], bins=20, alpha=0.6,
                color=color, label=f"Target={label}")
    ax.set_title(col)
    ax.set_xlabel(col)
    ax.legend(fontsize=7)

fig.suptitle("Key Feature Distributions by Target", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# --- Categorical features: proportion of disease-positive cases per category ---
# Each bar shows what fraction of patients in that category have heart disease.
# Chest pain type (cp) and exercise-induced angina (exang) show strong separation.
cat_features = ["sex", "cp", "fbs", "restecg", "exang", "slope", "ca", "thal"]
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for ax, col in zip(axes.flatten(), cat_features):
    ct = pd.crosstab(df[col], df["target"], normalize="index")
    ct.plot(kind="bar", ax=ax, color=["steelblue", "tomato"],
            edgecolor="white", legend=False)
    ax.set_title(f"{col} vs target")
    ax.set_xlabel(col)
    ax.set_ylabel("Proportion")
    ax.tick_params(axis="x", rotation=0)

handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in ["steelblue", "tomato"]]
fig.legend(handles, ["No Disease", "Heart Disease"], loc="lower right", ncol=2)
plt.suptitle("Categorical Features vs Target", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# --- Correlation heatmap ---
# Shows linear correlations between all features. Strong correlations with
# `target` suggest predictive value. High inter-feature correlations could
# cause multicollinearity issues for linear models.
fig, ax = plt.subplots(figsize=(12, 9))
corr = processor.correlation_matrix()
mask = np.triu(np.ones_like(corr, dtype=bool))  # hide upper triangle (redundant)
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, linewidths=0.5, ax=ax)
ax.set_title("Feature Correlation Matrix", fontsize=13)
plt.tight_layout()
plt.show()

---
## 6. Model Training

We split the data into training (80%) and test (20%) sets using stratified sampling so both splits have the same class balance. Then `ModelTrainer` from `src/model_training.py` builds and trains three scikit-learn pipelines:

- **Logistic Regression** — a simple, interpretable linear baseline
- **Random Forest** — an ensemble of decision trees that captures non-linear relationships
- **Decision Tree** — a single tree included to show the value of ensembling: comparing it against Random Forest demonstrates how much accuracy is gained by combining many trees instead of one

After training, all model files and evaluation metrics are saved to `models/` so the Streamlit app can load them without re-training.

In [ ]:
# Split into features (X) and target (y), then train/test split
X, y = processor.get_features_and_target()
X_train, X_test, y_train, y_test = processor.split_data(test_size=0.2, random_state=42)

print(f"Train size : {len(X_train)} samples  |  Test size : {len(X_test)} samples")
print(f"Train positive rate : {y_train.mean():.1%}")
print(f"Test  positive rate : {y_test.mean():.1%}  (stratification check)")

In [ ]:
# Build pipelines, train all three models, and evaluate on the held-out test set
trainer = ModelTrainer(random_state=42)
trainer.build_pipelines()
trainer.train_models(X_train, y_train)
results = trainer.evaluate(X_test, y_test)

print("\n=== Model Comparison ===")
print(trainer.compare().to_string())

In [ ]:
# Save all artefacts needed by the Streamlit app:
#   models/logistic_regression.pkl  — loaded by app.py
#   models/random_forest.pkl        — loaded by app.py
#   models/training_results.json    — read by the Model Performance page

MODELS_DIR.mkdir(exist_ok=True)

# Save best model via ModelTrainer's built-in method
trainer.save_best_model(str(MODEL_PATH))

# Save individual .pkl files (app.py looks for these by model slug)
for name, result in results.items():
    slug = name.lower().replace(" ", "_")
    joblib.dump(result.pipeline, MODELS_DIR / f"{slug}.pkl")

# Build and save training_results.json
results_dict = {
    name: {
        "accuracy":  round(r.accuracy, 4),
        "f1":        round(r.f1, 4),
        "precision": round(r.precision, 4),
        "recall":    round(r.recall, 4),
        "roc_auc":   round(r.roc_auc, 4),
    }
    for name, r in results.items()
}
with RESULTS_PATH.open("w") as f:
    json.dump(results_dict, f, indent=2)

print(f"✅ Models saved  : {MODELS_DIR}")
print(f"✅ Results saved : {RESULTS_PATH}")

### Model Results — EC4 vs EC3

EC4 uses the Cleveland dataset with all 13 features (including `ca` and `thal`).
Results on 20% held-out test split, 303 rows:

| Model | EC3 Accuracy | EC4 Accuracy | Notes |
|-------|-------------|-------------|-------|
| Logistic Regression | 0.869 | 0.803 | ↓ smaller dataset |
| Decision Tree | 0.754 | 0.803 | ↑ tied with LR |
| Random Forest | 0.885 | 0.754 | ↓ needs more data |

With only 303 rows, Logistic Regression and Decision Tree tie at 80.3%.
Random Forest typically needs more data to generalise well.
See `scripts/investigate_datasets.py` for the full dataset investigation.

---
## 7. Evaluation & Visualisations

We evaluate both models on the test set using five metrics:

- **Accuracy** — overall fraction of correct predictions
- **Precision** — of all predicted positives, how many are truly positive
- **Recall** — of all actual positives, how many did the model catch
- **F1-score** — harmonic mean of precision and recall
- **ROC AUC** — area under the ROC curve; measures ranking quality across all thresholds

In a medical context recall is especially important: missing a true disease case (false negative) is more costly than a false alarm.

In [ ]:
# --- Side-by-side bar chart of all five metrics for both models ---
metrics = ["accuracy", "f1", "precision", "recall", "roc_auc"]
x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(11, 5))
for i, (name, color) in enumerate(zip(results_dict.keys(), ["steelblue", "tomato", "seagreen"])):
    vals = [results_dict[name][m] for m in metrics]
    bars = ax.bar(x + i * width, vals, width, label=name, color=color, alpha=0.85)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, v + 0.005,
                f"{v:.3f}", ha="center", va="bottom", fontsize=8)

ax.set_xticks(x + width / 2)
ax.set_xticklabels([m.replace("_", " ").title() for m in metrics])
ax.set_ylim(0, 1.08)
ax.set_ylabel("Score")
ax.set_title("Model Comparison — All Metrics")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- Confusion matrices ---
# Each cell shows: top-left = true negatives, top-right = false positives,
# bottom-left = false negatives (missed disease), bottom-right = true positives.
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, result) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, result.pipeline.predict(X_test))
    disp = ConfusionMatrixDisplay(cm, display_labels=["No Disease", "Heart Disease"])
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(f"Confusion Matrix — {name}")

plt.tight_layout()
plt.show()

In [ ]:
# --- ROC curves ---
# The ROC curve plots true positive rate vs false positive rate at every
# decision threshold. A higher AUC means better discrimination between classes.
fig, ax = plt.subplots(figsize=(8, 6))

for (name, result), color in zip(results.items(), ["steelblue", "tomato", "seagreen"]):
    if hasattr(result.pipeline, "predict_proba"):
        y_score = result.pipeline.predict_proba(X_test)[:, 1]
    else:
        y_score = result.pipeline.decision_function(X_test)
    fpr, tpr, _ = roc_curve(y_test, y_score)
    auc = roc_auc_score(y_test, y_score)
    ax.plot(fpr, tpr, color=color, lw=2, label=f"{name} (AUC = {auc:.3f})")

ax.plot([0, 1], [0, 1], "k--", lw=1, label="Random classifier")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

In [ ]:
# --- Random Forest feature importances ---
# Feature importance shows which input variables the forest relies on most.
# This helps explain the model and highlights clinically meaningful predictors.
rf_result = results.get("Random Forest")
if rf_result is not None:
    pipeline = rf_result.pipeline
    # Try common step names; fall back to the last step in the pipeline
    clf = (
        pipeline.named_steps.get("classifier")
        or pipeline.named_steps.get("model")
        or list(pipeline.named_steps.values())[-1]
    )
    if hasattr(clf, "feature_importances_"):
        feature_names = X.columns.tolist()
        importances = clf.feature_importances_
        indices = np.argsort(importances)[::-1]

        fig, ax = plt.subplots(figsize=(10, 5))
        ax.bar(range(len(feature_names)),
               importances[indices], color="steelblue", edgecolor="white")
        ax.set_xticks(range(len(feature_names)))
        ax.set_xticklabels([feature_names[i] for i in indices], rotation=45, ha="right")
        ax.set_title("Random Forest — Feature Importances")
        ax.set_ylabel("Importance (mean decrease in impurity)")
        plt.tight_layout()
        plt.show()
    else:
        print("Feature importances not available for this pipeline configuration.")
else:
    print("Random Forest result not found in results dict.")

In [ ]:
# --- Full classification reports ---
# Per-class precision, recall, and F1. Useful for checking if one class
# is being systematically under-predicted.
for name, result in results.items():
    print(f"\n{'='*52}")
    print(f" Classification Report — {name}")
    print(f"{'='*52}")
    print(classification_report(
        y_test,
        result.pipeline.predict(X_test),
        target_names=["No Disease", "Heart Disease"]
    ))

---
## 8. Streamlit App — Launch & Inline Results

The Streamlit app (`app.py`) provides a browser-based interface with four pages: **Home**, **Prediction**, **Model Performance**, and **About**.

The cell below starts the app as a background process (using the same Python interpreter as this notebook, so it shares the `.venv`). It then waits up to 20 seconds for the server to become ready before embedding it inline via an `IFrame`.

The subsequent cells reproduce the **Model Performance** page charts directly in the notebook using the `training_results.json` saved during training — so the results are visible here even if the iframe does not render in your environment (e.g. VS Code).

**To open the app manually instead**, run in a terminal:
```bash
python src/main.py --streamlit
```

In [ ]:
STREAMLIT_PORT = 8501

def is_port_open(port, host="localhost"):
    """Return True if a process is already listening on the port."""
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(1)
        return s.connect_ex((host, port)) == 0

if is_port_open(STREAMLIT_PORT):
    print(f"Streamlit is already running on port {STREAMLIT_PORT}.")
else:
    app_path = str(Path(PROJECT_ROOT) / "app.py")
    _proc = subprocess.Popen(
        [
            sys.executable, "-m", "streamlit", "run", app_path,
            "--server.port", str(STREAMLIT_PORT),
            "--server.headless", "true",
            "--server.runOnSave", "false",
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    print("Starting Streamlit", end="")
    for _ in range(20):
        time.sleep(1)
        print(".", end="", flush=True)
        if is_port_open(STREAMLIT_PORT):
            print(" ready!")
            break
    else:
        print("\n⚠️  Timed out — check that streamlit is installed in your environment.")

STREAMLIT_URL = f"http://localhost:{STREAMLIT_PORT}"
print(f"\nApp URL: {STREAMLIT_URL}")

In [ ]:
# Embed the running Streamlit app inline.
# Works in classic Jupyter Notebook and most JupyterLab configurations.
# In VS Code notebooks the IFrame may be blocked — use the link instead.
display(HTML(f"""
<div style="border:2px solid #e2e8f0; border-radius:8px; padding:12px;
            margin:8px 0; background:#f8fafc; font-family:sans-serif">
  <strong>❤️ Heart Disease Risk Predictor — Streamlit App</strong><br><br>
  <a href="{STREAMLIT_URL}" target="_blank" style="color:#1d4ed8">
    Open in a new tab → {STREAMLIT_URL}
  </a>
  <p style="color:#64748b; font-size:0.9em; margin-top:8px">
    Use the sidebar to navigate between Home, Prediction, Model Performance, and About.
  </p>
</div>
"""))

display(IFrame(src=STREAMLIT_URL, width="100%", height=700))

### Streamlit Model Performance page — replicated inline

The cells below reproduce the exact charts from the **Model Performance** page of the Streamlit app. They read from `models/training_results.json`, the same file the app reads, so the numbers are always in sync.

In [ ]:
# Load the saved results and display as a table (mirrors the Streamlit dataframe widget)
with RESULTS_PATH.open() as f:
    saved_results = json.load(f)

results_display = pd.DataFrame(saved_results).T.round(4)
results_display.columns = [c.replace("_", " ").title() for c in results_display.columns]
results_display.index.name = "Model"

print("=== Model Performance Table (Streamlit — Model Performance page) ===")
results_display

In [ ]:
# Row 1 of Streamlit charts: Accuracy and ROC AUC
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, metric in zip(axes, ["accuracy", "roc_auc"]):
    vals = {name: saved_results[name][metric] for name in saved_results}
    bars = ax.bar(vals.keys(), vals.values(),
                  color=["steelblue", "tomato", "seagreen"], edgecolor="white", alpha=0.85)
    for bar, v in zip(bars, vals.values()):
        ax.text(bar.get_x() + bar.get_width() / 2, v + 0.005,
                f"{v:.4f}", ha="center", fontsize=9)
    ax.set_title(metric.replace("_", " ").title())
    ax.set_ylim(0, 1.08)
    ax.set_ylabel("Score")

fig.suptitle("Streamlit — Model Performance page (row 1)", fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Row 2 of Streamlit charts: Precision, Recall, F1
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, metric in zip(axes, ["precision", "recall", "f1"]):
    vals = {name: saved_results[name][metric] for name in saved_results}
    bars = ax.bar(vals.keys(), vals.values(),
                  color=["steelblue", "tomato", "seagreen"], edgecolor="white", alpha=0.85)
    for bar, v in zip(bars, vals.values()):
        ax.text(bar.get_x() + bar.get_width() / 2, v + 0.005,
                f"{v:.4f}", ha="center", fontsize=9)
    ax.set_title(metric.title())
    ax.set_ylim(0, 1.08)
    ax.set_ylabel("Score")

fig.suptitle("Streamlit — Model Performance page (row 2)", fontsize=11)
plt.tight_layout()
plt.show()

---
## 9. Ethical Reflection

*(Swedish as per course requirements)*

### Dataset

Datasetet kommer från UCI Heart Disease och innehåller 13 kliniska egenskaper såsom ålder, kön, blodtryck, kolesterol, maxpuls och thalassemia. Målvariabeln `target` anger om patienten har hjärtsjukdom (1) eller inte (0).

### Modellval

Två modeller tränades: Logistic Regression och Random Forest. Random Forest valdes som bästa modell baserat på högre noggrannhet och ROC AUC. Random Forest fångar icke-linjära samband och är robust för blandade numeriska och kategoriska variabler.

### Resultat

Modellerna utvärderades på ett stratifierat test-set (20%). Random Forest uppnådde bättre resultat på samtliga mätetal — se tabellen och diagrammen ovan.

### Etiskt resonemang

Modelldatabasen representerar patienter där fördelningen mellan åldrar och kön kan vara sned. Prediktionsmodeller för hjärtsjukdom bör därför inte användas som ensam beslutsgrund i vården. Det är viktigt att modellen kompletteras med kliniskt omdöme och diagnostiska tester. Felaktiga prediktioner kan leda till onödig oro, överbehandling eller att en patient missas.

Dessutom kan modellen förstärka befintliga biaser om datasetet inte inkluderar tillräckligt många underrepresenterade grupper. Därför är det etiskt viktigt att tydligt kommunicera modellens begränsningar och använda den som stöd, inte ersättning, till medicinsk expertis.

**Sammanfattning:** Modellen är ett pedagogiskt verktyg — inte ett medicinskt diagnostikinstrument.

---
## 12. CI/CD — GitHub Actions

The project includes a GitHub Actions workflow that automatically runs the full test suite on every push and pull request. This ensures that no commit breaks the 44 tests or drops coverage below 85%.

The workflow file lives at `.github/workflows/tests.yml` in the repository. Its contents:

```yaml
name: Tests

on: [push, pull_request]

jobs:
  test:
    runs-on: ubuntu-latest

    steps:
      - name: Check out repository
        uses: actions/checkout@v4

      - name: Set up Python 3.11
        uses: actions/setup-python@v5
        with:
          python-version: "3.11"

      - name: Install dependencies
        run: pip install -r requirements.txt

      - name: Run tests with coverage
        run: pytest tests/ --cov=src --cov-report=term-missing --cov-fail-under=85

      - name: Upload coverage report
        if: always()
        uses: actions/upload-artifact@v4
        with:
          name: coverage-report
          path: htmlcov/
```

**To activate it**, place the file at `.github/workflows/tests.yml` and push to GitHub — Actions picks it up automatically. You can see run results under the **Actions** tab in your repository.

To run the same tests locally at any time:
```bash
pytest tests/ -v
pytest tests/ --cov=src --cov-report=html   # generates htmlcov/index.html
```

---
## 11. Terminal App

The project includes an interactive terminal app (`src/terminal_app.py`) built around the `HeartApp` class. It has four components:

| Component | Description |
|-----------|-------------|
| `FeatureInfo` | Dataclass holding metadata (name, description, data type) for each of the 13 clinical features |
| `__init__` | Loads the trained model from disk and defines all 13 features with descriptions |
| `prompt_for_inputs()` | Interactively prompts the user for each feature value, validates input type, and retries on invalid input |
| `predict()` | Takes a list of 13 values, runs inference and returns the binary prediction and probability |
| `run()` | Main loop: calls `prompt_for_inputs()`, then `predict()`, prints the result, and asks if the user wants to assess another patient |

The cells below demonstrate each component programmatically (without interactive input) so they can run in the notebook.

To run the full interactive version in a terminal:
```bash
python src/main.py --app
```

In [ ]:
# --- FeatureInfo and feature list ---
# Show all 13 features with their descriptions and expected data types.
# This is what prompt_for_inputs() uses to guide the user.
from src.terminal_app import HeartApp, FeatureInfo

app = HeartApp(str(MODEL_PATH))

print(f"{'Feature':<12} {'Type':<8} {'Description'}")
print("-" * 60)
for f in app.features:
    print(f"{f.name:<12} {f.data_type:<8} {f.description}")

In [ ]:
# --- predict() ---
# Demonstrate predict() with three example patients.
# In the real app these values come from prompt_for_inputs().

patients = {
    "Low risk patient": [45, 0, 1, 130, 250, 1, 0, 160, 1, 2.0, 2, 1, 3],
    "High risk patient": [65, 1, 0, 150, 300, 0, 2, 100, 0, 0.5, 1, 2, 7],
    "Borderline patient": [55, 1, 2, 120, 200, 0, 1, 140, 0, 1.0, 1, 0, 3],
}

print(f"{'Patient':<22} {'Result':<10} {'Confidence'}")
print("-" * 45)
for label, values in patients.items():
    prediction, probability = app.predict(values)
    risk = "LIKELY" if prediction == 1 else "UNLIKELY"
    print(f"{label:<22} {risk:<10} {probability:.1%}")

In [ ]:
# --- run() — full loop simulation ---
# Simulate the complete run() loop with mocked input
# to show the full user experience without blocking the notebook.
from unittest.mock import patch

# Simulate: one patient assessment, then exit
mock_inputs = [
    '55', '1', '0', '120', '200', '0', '1', '140', '0', '1.0', '1', '0', '3',  # feature values
    'n'  # exit
]

with patch('builtins.input', side_effect=mock_inputs):
    app.run()

---
## Summary

| Step | What happened |
|------|---------------|
| Environment check | Confirmed `.venv` kernel is active |
| Data loading | UCI Heart Disease CSV loaded and cleaned via `DataProcessor` |
| EDA | Target balance, feature distributions, correlations visualised |
| Training | Logistic Regression, Random Forest and Decision Tree trained on 80% split |
| Saving | `.pkl` model files and `training_results.json` written to `models/` |
| Evaluation | Confusion matrices, ROC curves, feature importances, classification reports |
| Streamlit | App launched on port 8501, embedded inline, charts replicated in notebook |
| CI/CD | GitHub Actions workflow runs tests on every push (`tests.yml`) |

To run the terminal prediction app:
```bash
python src/main.py --app
```